# Y_tx-conv — Taxa de Conversão para Agricultura e Pastagem

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** taxa de conversão para agricultura e pastagem, proporção 0–1, municipal, anual*  
**Conceito e Mecanismo Econômico:** Mede a taxa anual de conversão (0 a 1), definida pela divisão entre a área convertida no ano t e o estoque de vegetação nativa remanescente no início do período (t-1).  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h2__native_veg__mapbiomas_col10_1__candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Construir: área convertida por destino / área nativa no início do período.
> - Este arquivo fornece o denominador candidato (native_veg_ha_lag1), não o numerador.
> - Compatibilizar geografia e máscara com as transições; denominador zero implica taxa indefinida.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do IBGE do município (7 dígitos). Base de 141 municípios.
- **`municipio`**: Nome do município.
- **`ano`**: Ano da observação.
- **`area_mapeada_ha`**: Área territorial total mapeada do município (ha).
- **`native_veg_ha_it`**: Estoque de vegetação nativa no ano t (ha).
- **`native_veg_share_it`**: Proporção de vegetação nativa no ano t (0 a 1).
- **`native_veg_ha_lag1`**: Estoque nativo no ano t-1 (DENOMINADOR CANDIDATO da taxa de conversão).
- **`native_veg_share_lag1`**: Fração de vegetação nativa no ano t-1.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h2__native_veg__mapbiomas_col10_1__candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `Y_tx-conv`:
- **Atenção:** Construir: área convertida por destino / área nativa no início do período.
- **Atenção:** Este arquivo fornece o denominador candidato (native_veg_ha_lag1), não o numerador.
- **Atenção:** Compatibilizar geografia e máscara com as transições; denominador zero implica taxa indefinida.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')

for col in ['area_mapeada_ha', 'native_veg_ha_it', 'native_veg_share_it', 'native_veg_ha_lag1', 'native_veg_share_lag1']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# Se houver linhas remapeadas, aglutinar somando hectares e recalculando proporções
df = df.groupby(['codigo_ibge', 'ano'], as_index=False).agg({
    'area_mapeada_ha': 'sum',
    'native_veg_ha_it': 'sum',
    'native_veg_ha_lag1': 'sum'
})
df['native_veg_share_it'] = df['native_veg_ha_it'] / df['area_mapeada_ha']
df['native_veg_share_lag1'] = df['native_veg_ha_lag1'] / df['area_mapeada_ha']

print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva do estoque nativo no início do período (native_veg_ha_lag1):")
print(df['native_veg_ha_lag1'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# REGRA DO TXT: 'denominador zero implica taxa indefinida'
den_zeros = (df['native_veg_ha_lag1'] == 0).sum()
den_nulos = df['native_veg_ha_lag1'].isna().sum()
print(f"Denominadores iguais a zero: {den_zeros} | Denominadores ausentes: {den_nulos}")

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['native_veg_share_lag1'].dropna(), bins=30, kde=True, ax=axes[0], color='teal')
axes[0].set_title("Distribuição da Fração Nativa (t-1)")

sns.boxplot(y=df['native_veg_share_lag1'], ax=axes[1], color='powderblue')
axes[1].set_title("Boxplot da Fração Nativa (t-1)")

df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
sns.lineplot(data=df_painel, x='ano', y='native_veg_share_lag1', marker='o', ax=axes[2], color='darkcyan')
axes[2].set_title("Média Municipal da Fração Nativa (2001-2024)")

plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
# Trajetória temporal do estoque total remanescente
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_painel, x='ano', y='native_veg_ha_it', estimator='sum', color='darkgreen')
plt.title("Estoque Total Agregado de Vegetação Nativa em Mato Grosso (2001-2024)")
plt.ylabel("Hectares de Nativa")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Fase 6: Feature Engineering — Construção da Variável Y_tx-conv (Taxa de Conversão)

### 1. Conceito e Justificativa Econométrica
- **O que representa:** Mede a proporção anual (taxa de 0 a 1) da vegetação nativa remanescente que foi convertida em novos usos agropecuários em cada município $i$ no ano $t$.
- **Fórmula Matemática:**
  $$Y\_tx\_conv_{i, t} = \frac{\text{Área Convertida}_{i, t}}{\text{Estoque Nativo Remanescente}_{i, t-1}}$$
- **Por que defasar o denominador ($t-1$):** A taxa de conversão mede a fração do estoque que existia *no início do período* de decisão que foi consumida pela expansão. O estoque contemporâneo ($t$) já reflete a perda, gerando viés mecânico de divisão.
- **Salvaguarda de Denominador Quase Zero (Ressalva Matemática Crucial):**
  Se uma cidade já perdeu quase toda a sua mata e restam apenas 50 hectares (`native_veg_ha_lag1 <= 100`), qualquer pequena clareira de 25 ha geraria uma taxa aparente de 50% ($0{,}50$). Pior: se restasse 1 ha e fossem convertidos 2 ha (por margem de erro cartográfico), a taxa explodiria para 200% ($2{,}0$), distorcendo totalmente a regressão!
  Portanto:
  1. Se `native_veg_ha_lag1 <= 100 ha`, a taxa é definida como `NaN`.
  2. Criamos uma dummy indicadora de fronteira exaurida (`flag_fronteira_fechada = 1` se `share_nativo < 5%`).
  3. Truncamos a taxa ao limite superior de $1{,}0$ ($100\%$).


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Selecionar as colunas essenciais para o cálculo da taxa
df_taxa = df[['codigo_ibge', 'ano', 'native_veg_ha_it', 'native_veg_ha_lag1', 'native_veg_share_lag1']].copy()

# 2. Simular/integrar a área convertida no ano t (numerador da taxa)
# Em ambiente integrado, este valor é herdado da variável Y_conv (hectares convertidos no ano t)
# Aqui usamos a perda anual de mata nativa como proxy da conversão bruta para demonstração:
df_taxa.sort_values(['codigo_ibge', 'ano'], inplace=True)
df_taxa['perda_nativa_ha'] = (df_taxa['native_veg_ha_lag1'] - df_taxa['native_veg_ha_it']).clip(lower=0)

# 3. SALVAGUARDA MATEMÁTICA: Denominador pequeno gera explosão numérica artificial!
# Se o estoque de vegetação nativa no ano anterior for menor ou igual a 100 ha, a taxa é indefinida (NaN).
LIMITE_MINIMO_NATIVO_HA = 100.0

# native_veg_ha_lag1: estoque de vegetação nativa no ano anterior (ha) -> denominador
# perda_nativa_ha: área de floresta convertida no ano corrente (ha) -> numerador
# Y_tx_conv: taxa anual de conversão de vegetação nativa (proporção de 0 a 1)
df_taxa['Y_tx_conv'] = np.where(
    df_taxa['native_veg_ha_lag1'] > LIMITE_MINIMO_NATIVO_HA,
    df_taxa['perda_nativa_ha'] / df_taxa['native_veg_ha_lag1'],
    np.nan
)

# Truncar ao teto de 1.0 (não é possível converter mais de 100% da floresta remanescente em um único ano)
df_taxa['Y_tx_conv'] = df_taxa['Y_tx_conv'].clip(lower=0.0, upper=1.0)

# 4. Variável de Controle Institucional / Estrutural:
# flag_fronteira_fechada: vale 1 se o município tiver menos de 5% de vegetação nativa restante
df_taxa['flag_fronteira_fechada'] = (df_taxa['native_veg_share_lag1'] < 0.05).astype(int)

# 5. Filtrar o painel 2001-2024
painel_tx = df_taxa[(df_taxa['ano'] >= 2001) & (df_taxa['ano'] <= 2024)].copy()

print(f"Total de observações no painel da taxa: {len(painel_tx)}")
print(f"Casos com denominador protegido (NaN): {painel_tx['Y_tx_conv'].isna().sum()}")
print(f"Municípios com fronteira praticamente fechada (share nativo < 5%): {painel_tx['flag_fronteira_fechada'].sum()}")
print("\nResumo da taxa de conversão Y_tx_conv:")
print(painel_tx[['Y_tx_conv', 'native_veg_ha_lag1', 'native_veg_share_lag1', 'flag_fronteira_fechada']].describe())